# 06 Large Data Collection: 5000 Papers

This notebook expands the ScholarSynth AI dataset to at least 5000 raw papers. It uses a larger topic bank, appends to the existing corpus, checkpoints after every topic, deduplicates papers, and regenerates processed chunks, train/validation/test splits, and fine-tuning JSONL files.

Expected runtime can be long because the notebook calls arXiv and Semantic Scholar repeatedly.

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Could not locate project root containing src/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.large_data_collection import collect_large_dataset, regenerate_derived_files
from src.topic_bank import LARGE_SCALE_TOPICS

print(f'Project root: {PROJECT_ROOT}')
print(f'Topics available: {len(LARGE_SCALE_TOPICS)}')

## Collection Settings

In [ ]:
TARGET_PAPERS = 5000
ARXIV_LIMIT = 120
SEMANTIC_LIMIT = 30
SLEEP_SECONDS = 1.0
SEMANTIC_API_KEY = None  # Optional. Add a key here if you have one.
APPEND_EXISTING = True

print('Target papers:', TARGET_PAPERS)
print('arXiv per topic:', ARXIV_LIMIT)
print('Semantic Scholar per topic:', SEMANTIC_LIMIT)

## Run Collection

This writes checkpoints to `data/raw_papers_checkpoint.csv` after every topic and final data to `data/raw_papers.csv`.

In [ ]:
raw_df = collect_large_dataset(
    target_papers=TARGET_PAPERS,
    arxiv_limit=ARXIV_LIMIT,
    semantic_limit=SEMANTIC_LIMIT,
    sleep_seconds=SLEEP_SECONDS,
    semantic_api_key=SEMANTIC_API_KEY,
    append_existing=APPEND_EXISTING,
)

raw_df.shape

## Regenerate Processed Data and Fine-Tuning Files

In [ ]:
regenerate_derived_files(raw_df)

## Final Counts

In [ ]:
import pandas as pd

files = [
    'data/raw_papers.csv',
    'data/processed_papers.csv',
    'data/train.csv',
    'data/val.csv',
    'data/test.csv',
    'data/finetune_dataset.jsonl',
    'data/finetune_train.jsonl',
    'data/finetune_val.jsonl',
    'data/finetune_test.jsonl',
]

rows = []
for file in files:
    path = PROJECT_ROOT / file
    line_count = sum(1 for _ in path.open('r', encoding='utf-8')) if path.exists() else 0
    rows.append({'file': file, 'lines': line_count, 'size_mb': path.stat().st_size / (1024 * 1024) if path.exists() else 0})

pd.DataFrame(rows)